# Testing Intern

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import pipeline

### Load in Data

In [ ]:
### LOAD IN DATA ###
import pandas as pd

demo_data_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/demo_set.csv"
demo_df = pd.read_csv(demo_data_path)

test_data_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/testing_set.csv"
testing_df = pd.read_csv(test_data_path)

In [ ]:
testing_df.head()

### Load in Model

In [ ]:
# 1. Build the pipeline
pipe = pipeline("image-text-to-text", model="OpenGVLab/InternVL3-8B-hf")

In [ ]:
# Define start of prompt
start_of_prompt = '''
STEP 1: How does this painting make you primarily feel? Select from the following: Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, Something-Else

then...

STEP 2: Give a detailed description (at least 7 words) about WHY you feel like this, based on SPECIFIC details of the painting.

Examples of GOOD descriptions:
  - "the sky looks gloomy and the shadows are scary"
  - "the red marks on the table look like drops of blood" (analogies are good!)
  - "the blue color of the lake contrasts with the orange hats of the men"

Examples of BAD descriptions:
  - Vague descriptions that do not explain WHY you felt like this:
    - "I like its colors" (be more specific)
    - "It is amazing, nice work! great painting" (why is it amazing? be specific)
    - "I don't know what this is" (try and explain what it looks like)

Note: if you selected "Something-Else" also explain how you felt in addition to why you felt Something-Else.

Finally, **output your response strictly in the following JSON format**:
{
    "emotion": string,
    "utterance": string
}

Please make sure that once you have fully thought through your answers and you respond with them in the above JSON Format and only output that JSON.
'''

In [ ]:
### Select Random Images For Demonstrations ###
def get_random_images(df, n):
    # Sample n rows at once
    random_rows = df.sample(n=n, replace=False)

    demos = []

    for i in range(n):
        row = random_rows.iloc[i]

        art_style = row["art_style"].lower()
        painting = row["painting"].lower()

        image_path = (
            f"/content/drive/Shareddrives/LLMs_Art_Project/Data/"
            f"{art_style}_images/{painting}.jpeg"
        )

        demos.append((image_path, row))

    return demos

In [ ]:
import json

def parse_model_output(output_text):
    try:
        # Extract only the JSON portion the model produced
        json_start = output_text.find("{")
        json_end = output_text.rfind("}") + 1
        json_str = output_text[json_start:json_end]
        data = json.loads(json_str)

        emotion = data.get("emotion", "")
        utterance = data.get("utterance", "")
        return emotion, utterance

    except Exception as e:
        print("JSON parse error:", e)
        return "", ""  # fallback if model output isn't valid JSON


In [ ]:
def run_intern(messages, max_new_tokens=200):
    """
    Run inference on Intern using a pre-formatted messages list.
    `messages` must be a list of dicts in the specified chat format.
    """
    # 3. Run inference
    outputs = pipe(text=messages, max_new_tokens=200, return_full_text=False)
    caption = outputs[0]["generated_text"]

    return caption


In [ ]:
from numpy import number
# Loop through testing dataframe (aka run one inference on each testing image)
for number_demos in [0, 1, 2, 3, 4]:
    results = []

    for index, row in testing_df.iterrows():
        # Extract test sample info
        test_art_style = row["art_style"].lower()
        test_painting_name = row["painting"].lower()
        test_emotion_gt = row["emotion"]
        test_utterance_gt = row["utterance"]

        # Build full file path for the test image
        test_image_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/{test_art_style}_images/{test_painting_name}.jpeg"
        # print(f"Processing test image {test_painting_name}")

        # Build input message
        content = []
        # Start of prompt
        content.append({"type": "text", "text": start_of_prompt})

        demo_painting_names = []
        demo_utterances = []

        # Add demos into prompt
        if number_demos > 0:
            # Set up demos
            content.append({"type": "text", "text": f"Here are {number_demos} examples of how to respond:"})

            # Get number_demos random (image_path, row) pairs
            demo_samples = get_random_images(demo_df, number_demos)

            # Unpack demo samples
            for i, (demo_image_path, demo_row) in enumerate(demo_samples, start=1):
                demo_emotion = demo_row["emotion"]
                demo_utterance = demo_row["utterance"]
                demo_painting = demo_row["painting"].lower()

                # Store for later
                demo_painting_names.append(demo_painting)
                demo_utterances.append(demo_utterance)

                # Add to prompt
                content.append({"type": "text", "text": f"Example {i}:"})
                content.append({"type": "image", "image": demo_image_path})
                content.append({"type": "text", "text": f"Emotion: {demo_emotion}\nReason: {demo_utterance}"})


        # Now add the actual test image
        content.append({"type": "text", "text": "Now analyze this painting."})
        content.append({"type": "image", "image": test_image_path})

        # Build the message
        messages = [
          {
              "role": "user",
              "content": content,
          }
        ]

        # Run the generation, note: switch function name per model
        generated_output = run_intern(messages)

        # Parse the JSON (predicted emotion + utterance)
        pred_emotion, pred_utterance = parse_model_output(generated_output)

        # STORE RESULTS
        # ------------------------
        results.append({
            "test_painting": test_painting_name,
            "demo_paintings": demo_painting_names,
            "demo_utterances": demo_utterances,
            "predicted_emotion": pred_emotion,
            "generated_description": pred_utterance
        })

        # Print out generated output
        #print(f'Generated output for test image {index} titled {test_painting_name}:')
       # print(f'{generated_output}')
        #print(f'Predicted emotion: {pred_emotion}')
        #print(f'Predicted utterance: {pred_utterance}')
        #print('')

    # Store results in a dataframe
    results_df = pd.DataFrame(results)
    # Build Output Path
    output_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Intern/intern_results_{number_demos}shot.csv"
    # Save to csv file
    results_df.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path}")


### Extra Code


In [ ]:
### BUILD MESSAGE TO PASS INTO FUNCTION ###

# Get image path and image descriptions for demonstrations
# Currently using 0 shot learning so no demonstrations needed
n = 2
demo_image_1 = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_chicago-6-1961.jpeg"
demo_description_1 = "The black sharp lines running throughout the composition is jarring and uneasy."
demo_image_2 = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_new-york-2-1948.jpeg"
demo_description_2 = "Jagged sharp edges of shapes make me feel fear"



# Get image path for testing image
test_image_path = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_acolman-1-1955.jpeg"




# Prepare message — image + text prompt
messages = [
    {
            "role": "user",
            "content": [
                {"type": "text", "text": text_prompt},
                {"type": "text", "text": f"Here are {n} examples of how to respond:"},
                {"type": "text", "text": "Example 1:"},
                {"type": "image", "image": demo_image_1},
                {"type": "text", "text": demo_description_1},
                {"type": "text", "text": "Example 2:"},
                {"type": "image", "image": demo_image_2},
                {"type": "text", "text": demo_description_2},
                {"type": "text", "text": "This is the art I want you to analyze"},
                {"type": "image", "image": test_image_path},
            ],
        }
    ]

In [ ]:
generated_output = run_intern(messages)

In [ ]:
from PIL import Image

image = Image.open(test_image_path).convert("RGB")
display(image)
print(generated_output)

In [ ]:
### EXAMPLE ###
'''
content = [
    {"type": "text", "text": text_prompt},
    {"type": "text", "text": f"First, however, I am going to show you {n} examples"},
    {"type": "text", "text": "Example 1:"},
    {"type": "image", "image": image_path1},
    {"type": "text", "text": "Example 2:"},
    {"type": "image", "image": image_path1},
    ...,
    {"type": "text", "text": "This is the art I want you to analyze"},
    {"type": "image", "image": image_path}
]
'''